# 01 — Exploratory Data Analysis

This notebook explores:
- BTC/USDT price history
- Mining production cost over time
- Price-to-Cost Ratio (PCR)
- Signal zone distribution
- Correlation between PCR and future returns

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from data.fetch_price import fetch_price_data
from data.fetch_onchain import fetch_hashrate
from models.production_cost import compute_production_cost
from signals.signal_engine import compute_signals, Signal

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Load data

In [ ]:
price_df = fetch_price_data()
hr_df = fetch_hashrate()
cost_df = compute_production_cost(hr_df)
signals_df = compute_signals(price_df, cost_df)

print(f"Price data: {len(price_df)} rows  {price_df.index[0].date()} – {price_df.index[-1].date()}")
print(f"Hashrate data: {len(hr_df)} rows")
print(f"Signals data: {len(signals_df)} rows")
signals_df.tail()

## 2. BTC Price vs Production Cost

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(signals_df.index, signals_df['close'], label='BTC Price', linewidth=1.2)
ax.semilogy(signals_df.index, signals_df['production_cost'], label='Production Cost (30d smooth)', linewidth=1.5, linestyle='--')
ax.set_title('BTC/USDT Price vs Mining Production Cost')
ax.set_ylabel('Price (USD, log scale)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Price-to-Cost Ratio (PCR)

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(14, 8))

# Upper: price
axes[0].semilogy(signals_df.index, signals_df['close'], color='steelblue', linewidth=1)
axes[0].set_ylabel('BTC Price (USD)')
axes[0].set_title('BTC Price and PCR Indicator')

# Lower: PCR with horizontal zone lines
pcr = signals_df['pcr']
axes[1].plot(signals_df.index, pcr, color='darkorange', linewidth=1)
axes[1].axhline(0.9, color='green', linestyle='--', linewidth=0.8, label='Strong Buy (0.9)')
axes[1].axhline(1.1, color='limegreen', linestyle='--', linewidth=0.8, label='Weak Buy (1.1)')
axes[1].axhline(2.0, color='red', linestyle='--', linewidth=0.8, label='Sell (2.0)')
axes[1].axhline(3.5, color='darkred', linestyle='--', linewidth=0.8, label='Strong Sell (3.5)')
axes[1].set_ylabel('PCR')
axes[1].legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Signal Zone Distribution

In [ ]:
zone_counts = signals_df['signal_zone'].value_counts()
zone_pct = (zone_counts / len(signals_df) * 100).round(1)
print("Signal zone distribution:")
print(zone_pct.to_string())

colors = {'STRONG_BUY': 'green', 'WEAK_BUY': 'limegreen', 'HOLD': 'steelblue', 'SELL': 'orange', 'STRONG_SELL': 'red'}
zone_counts.plot(kind='bar', color=[colors.get(z, 'grey') for z in zone_counts.index])
plt.title('Days per Signal Zone')
plt.ylabel('Days')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 5. PCR vs Forward Returns

In [ ]:
for horizon in [30, 90, 180]:
    fwd_ret = signals_df['close'].pct_change(horizon).shift(-horizon)
    corr = signals_df['pcr'].corr(fwd_ret)
    print(f"{horizon:>3}d forward return correlation with PCR: {corr:+.3f}")